# 01 - Understanding the IMPROVE/CSA Dataset

This notebook aims to explain, in a simple way, the structure of the dataset used in the Final Degree Project.

The project is based on a basic idea:

**gene expression of the cell line + molecular representation of the drug → AUC response**

Therefore, before training models such as DeepTTC or Random Forest, it is important to understand what each file contains, how the files are related to each other, and what each row of the problem represents.

In this notebook, the three main files are reviewed:

- `response.tsv`: contains the cell–drug pairs and the pharmacological response.
- `drug_SMILES.tsv`: contains the molecular representation of the drugs.
- `cancer_gene_expression.tsv`: contains the gene expression data of the cell lines.

The splits and the distribution of the target variable AUC are also reviewed.


## 1. Loading Libraries and Defining Paths

First, the required libraries are loaded and the main project paths are defined.

The notebook is designed to be executed from the `notebooks_tfg/` folder. For this reason, the project root is considered to be the previous folder (`..`). If the notebook is executed from another location, it is enough to modify the `ROOT` variable.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# If the notebook is inside notebooks_tfg/, the project root is the previous folder.
ROOT = Path("..").resolve()

response_path = ROOT / "csa_data/raw_data/y_data/response.tsv"
smiles_path = ROOT / "csa_data/raw_data/x_data/drug_SMILES.tsv"
gene_path = ROOT / "csa_data/raw_data/x_data/cancer_gene_expression.tsv"
splits_dir = ROOT / "csa_data/raw_data/splits"

print("ROOT:", ROOT)
print("response_path:", response_path)
print("smiles_path:", smiles_path)
print("gene_path:", gene_path)
print("splits_dir:", splits_dir)


## 2. Loading the Main Files

In this project, three sources of information are used:

1. **Pharmacological response**: indicates how a cell responds to a drug.
2. **Drug SMILES**: represents the chemical structure of the drug as text.
3. **Gene expression**: represents the molecular state of each cell line.

These three sources are later combined to build the training samples.


In [ ]:
response = pd.read_csv(response_path, sep="\t")
smiles = pd.read_csv(smiles_path, sep="\t")
genes = pd.read_csv(gene_path, sep="\t")

print("response:", response.shape)
print("smiles:", smiles.shape)
print("genes:", genes.shape)


## 3. First Look at the Data

The first rows of each file are displayed below. This allows us to quickly identify the main columns and the type of information stored in each file.


In [ ]:
response.head()


In [ ]:
smiles.head()


In [ ]:
genes.head()


## 4. Explanation of Each File

### `response.tsv`

Each row represents a combination between a cell line and a drug. The most important columns for this project are:

- `improve_sample_id`: identifier of the cell line.
- `improve_chem_id`: identifier of the drug.
- `auc`: pharmacological response to be predicted.

### `drug_SMILES.tsv`

This file links each drug to its molecular representation in SMILES format.

A SMILES string is a textual way of representing a molecule. Instead of using a chemical image, the drug structure is encoded as a sequence of characters.

### `cancer_gene_expression.tsv`

This file contains the gene expression matrix of the cell lines. Each row corresponds to a cell line, and each column represents a gene-related variable.


## 5. Available Columns

The main columns of each file are listed to check that the expected identifiers and variables are present.


In [ ]:
print("Response columns:")
print(response.columns.tolist())

print("\nSMILES columns:")
print(smiles.columns.tolist())

print("\nFirst gene expression columns:")
print(genes.columns.tolist()[:20])

print("\nTotal number of columns in gene expression:", len(genes.columns))


## 6. Checking Missing Values

Before training any model, it is useful to check whether there are missing values. Missing values can affect preprocessing, training, or metric computation.


In [ ]:
print("NaNs in response:", response.isna().sum().sum())
print("NaNs in smiles:", smiles.isna().sum().sum())
print("NaNs in genes:", genes.isna().sum().sum())


## 7. Identifying Key Columns

To make the analysis more robust, the columns related to the following elements are detected automatically:

- cell identifier;
- drug identifier;
- target variable AUC;
- SMILES representation.


In [ ]:
sample_cols = [c for c in response.columns if "sample" in c.lower() or "cell" in c.lower()]
drug_cols = [c for c in response.columns if "chem" in c.lower() or "drug" in c.lower()]
auc_cols = [c for c in response.columns if "auc" in c.lower()]
smiles_cols = [c for c in smiles.columns if "smiles" in c.lower()]
smiles_drug_cols = [c for c in smiles.columns if "chem" in c.lower() or "drug" in c.lower()]

print("Candidate cell columns:", sample_cols)
print("Candidate drug columns in response:", drug_cols)
print("Candidate AUC columns:", auc_cols)
print("Candidate SMILES columns:", smiles_cols)
print("Candidate drug columns in smiles:", smiles_drug_cols)

sample_col = sample_cols[0]
drug_col = drug_cols[0]
auc_col = auc_cols[0]
smiles_col = smiles_cols[0]
smiles_drug_col = smiles_drug_cols[0]

print("\nUsing:")
print("sample_col:", sample_col)
print("drug_col:", drug_col)
print("auc_col:", auc_col)
print("smiles_col:", smiles_col)
print("smiles_drug_col:", smiles_drug_col)


## 8. AUC Distribution

AUC is the target variable of the problem. The model attempts to predict this value for each cell–drug pair.

The AUC distribution is important because it helps us understand the range of values, the concentration of samples, and possible imbalances in the response variable.


In [ ]:
print(response[auc_col].describe())

plt.figure()
plt.hist(response[auc_col], bins=30)
plt.xlabel("AUC")
plt.ylabel("Frequency")
plt.title("Global AUC Distribution")
plt.tight_layout()
plt.show()


## 9. Number of Cell Lines, Drugs, and Pairs

Each example in the problem is a cell–drug pair. For this reason, we review how many cell lines, drugs, and combinations exist in the dataset.


In [ ]:
n_cells = response[sample_col].nunique()
n_drugs = response[drug_col].nunique()
n_pairs = len(response)

print("Number of cell lines:", n_cells)
print("Number of drugs:", n_drugs)
print("Number of cell–drug pairs:", n_pairs)


## 10. Relationship Between Response and SMILES

The `response.tsv` file contains the drug identifier, but not the SMILES string directly. Therefore, it can be joined with `drug_SMILES.tsv` using the drug identifier.

This merge allows us to see, for each cell–drug pair, the textual molecular structure of the drug.


In [ ]:
response_smiles = response[[sample_col, drug_col, auc_col]].merge(
    smiles[[smiles_drug_col, smiles_col]],
    left_on=drug_col,
    right_on=smiles_drug_col,
    how="left"
)

print("response + smiles:", response_smiles.shape)
print("SMILES not found:", response_smiles[smiles_col].isna().sum())

response_smiles.head()


## 11. Relationship Between Response and Gene Expression

Gene expression data is joined with the response data using the cell line identifier. This check allows us to verify that the cells present in `response.tsv` have an available gene expression vector.


In [ ]:
gene_sample_col = genes.columns[0]

response_genes = response[[sample_col, drug_col, auc_col]].merge(
    genes[[gene_sample_col]],
    left_on=sample_col,
    right_on=gene_sample_col,
    how="left"
)

print("Identifier column in genes:", gene_sample_col)
print("response + genes:", response_genes.shape)
print("Cells without gene expression:", response_genes[gene_sample_col].isna().sum())


## 12. Example of a Training Sample

A training sample is built by combining:

1. A cell line.
2. The gene expression vector of that cell line.
3. A drug.
4. The SMILES representation of the drug.
5. The observed AUC value for that pair.

Therefore:

**model input = gene expression + drug representation**

**model output = AUC**


In [ ]:
example = response_smiles.iloc[0]

print("Example of a cell–drug pair")
print("----------------------------")
print("Cell line:", example[sample_col])
print("Drug:", example[drug_col])
print("SMILES:", example[smiles_col])
print("AUC:", example[auc_col])


## 13. Reviewing the Splits

The splits define which examples are used for training, validation, and testing.

In the normal split, the cell–drug pairs in train, validation, and test are different. However, the same cells or the same drugs may appear in several subsets. For this reason, more demanding scenarios were also studied in the project:

- **cell-out**: test cells were not seen during training.
- **drug-out**: test drugs were not seen during training.


In [ ]:
split_files = sorted(splits_dir.glob("CCLE_split_0_*.txt"))

for f in split_files:
    with open(f, "r") as fh:
        lines = [line.strip() for line in fh if line.strip()]
    print(f.name, "->", len(lines), "lines")


## 14. Conclusions of the Exploratory Analysis

This initial analysis identifies the three main components of the problem:

- the pharmacological response (`response.tsv`);
- the molecular representation of the drug (`drug_SMILES.tsv`);
- the gene expression of the cell line (`cancer_gene_expression.tsv`).

Each sample in the problem is built from a cell–drug pair. The input is composed of the cell gene expression and the drug representation, and the output is the AUC value.

It is also observed that the type of split is essential for interpreting the results. The normal split evaluates new pairs, but not necessarily completely unseen drugs or cells. Therefore, the `cell-out` and `drug-out` scenarios are important to study the real generalization ability of the model.
